# Automated_FIRMS_GEE_Wildfire_Assessment
**Linghan Qi**

**Last updated** 07/16/2026

## Project Overview
This notebook provides an automated, end-to-end spatial data pipeline for wildfire damage assessment and ecological recovery monitoring. By integrating **NASA FIRMS** active fire data with **Google Earth Engine (GEE)** cloud computing, this tool dynamically delineates fire perimeters, quantifies burn severity, and models long-term vegetation recovery trajectories.

## Key Features & Methodology
1. **Automated Data Ingestion:** Connects to the NASA FIRMS API to extract historical active fire anomalies (VIIRS/MODIS) within user-defined spatiotemporal parameters.
2. **Burn Severity Mapping (dNBR):** Computes the differenced Normalized Burn Ratio using Sentinel-2 L2A high-resolution multispectral imagery. Evaluates severity based on the official **USGS FIREMON** reclassification standards.
3. **Advanced Spatial Masking:** Implements a strict dual-masking algorithm utilizing the USDOS Global LSIB (ocean masking) and JRC Global Surface Water dataset (inland water masking) to eliminate false-positive atmospheric and glint artifacts.
4. **Time-Series Recovery Trajectory:** Extracts 12-month post-fire Normalized Difference Vegetation Index (NDVI) exclusively within the burned perimeter to statistically quantify ecological resilience against pre-fire baselines.
5. **Interactive Visualization:** Renders dynamic, multi-layered spatial maps using `geemap`, featuring academic legends and interactive thresholding.


## How to Use
1. Run the **Environment Setup** cell to install dependencies (`geemap`, `geopandas`).
2. Follow the prompt in the authentication cell to grant Google Earth Engine access.
3. Modify the parameters in the **Configuration Panel** (BBOX, Dates) below to run the analysis for any specific wildfire event globally.

# Parameters for all following tasks

In [ ]:
from google.colab import userdata

def get_colab_secret(name, default=''):
    """Read an optional Colab secret without preventing the GEE-only workflow."""
    try:
        return userdata.get(name) or default
    except Exception:
        return default

MAP_KEY = get_colab_secret('FIRMS_MAP_KEY')
GEE_PROJECT_ID = get_colab_secret('GEE_PROJECT_ID', 'final-research-lq-gis')
SOURCE = 'VIIRS_SNPP_SP'            # Cannot use _NRT for historical data
START_DATE = '2025-01-07'
END_DATE = '2025-01-15'
BBOX = '-119.2,33.7,-117.8,34.5'


## Skip to **Calculate dNBR and display on map** if you don't want to download FIRMS data


Use the FIRMS API to fetch active fire data or historical fire data from NASA, save it as a CSV file or GeoDataFrame shapefile, and gracefully handle potential errors.

In [ ]:
import requests
import pandas as pd
import geopandas as gpd
import io
import time
from datetime import datetime, timedelta
import folium

## Fetch historical fire data from NASA FIRMS API in batches for a specified date range.

In [ ]:
def fetch_historical_firms_data(map_key, source, bbox, start_date_str, end_date_str):
    # Convert string dates to datetime objects for calculation
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

    if start_date > end_date:
        print("❌ Error: Start date cannot be later than end date!")
        return None

    all_dataframes = []
    current_start = start_date

    current_time_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{current_time_str}] Starting to fetch historical data from {start_date_str} to {end_date_str}...")

    # Set maximum day range constraint (NASA limit for _SP is 5 days)
    MAX_DAY_RANGE = 5

    # Loop to slice dates, pushing forward by a maximum of 5 days each iteration
    while current_start <= end_date:
        # Calculate the DAY_RANGE needed for this batch (ensuring it doesn't exceed 5)
        days_diff = (end_date - current_start).days + 1
        day_range = min(MAX_DAY_RANGE, days_diff)

        # Format the current start date as a URL parameter
        date_param = current_start.strftime("%Y-%m-%d")

        url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{source}/{bbox}/{day_range}/{date_param}"
        batch_end_date = (current_start + timedelta(days=day_range - 1)).strftime("%Y-%m-%d")
        print(f"⏳ Fetching data from {date_param} to {batch_end_date} (Span: {day_range} days)...")

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            csv_data = io.StringIO(response.text)
            df = pd.read_csv(csv_data)

            if not df.empty:
                all_dataframes.append(df)
            else:
                print(f"⚠️ Warning: No fire points detected during this specific time frame.")

        except requests.exceptions.HTTPError as errh:
            print(f"❌ HTTP Error (If 429 Too Many Requests, rate limit was triggered): {errh}")
        except Exception as err:
            print(f"❌ An unexpected error occurred: {err}")

        # Push forward and update the start date for the next loop iteration
        current_start += timedelta(days=day_range)

        # Protect the API Key: pause for 2 seconds after each request to avoid being blocked by NASA
        time.sleep(2)

    # Merge all dataframes after the loop ends
    if all_dataframes:
        final_df = pd.concat(all_dataframes, ignore_index=True)
        # Remove duplicates based on latitude, longitude, and time to ensure data purity
        final_df = final_df.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time'])
        print(f"🎉 Successfully completed! Fetched and merged {len(final_df)} historical fire records.")
        return final_df
    else:
        print("⚠️ No valid data obtained within the entire specified date range.")
        return None

## Fetch active fire data from NASA FIRMS API based on the provided parameters. Returns a Pandas DataFrame if successful, or None if an error occurs.

In [ ]:
def fetch_firms_data(map_key, source, day_range, bbox):
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{source}/{bbox}/{day_range}"

    current_time_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{current_time_str}] Fetching data from NASA FIRMS...")

    try:
        # Send the network request with a 30-second timeout
        response = requests.get(url, timeout=30)
        # Raise an HTTPError if the status code is not 200
        response.raise_for_status()

        # Parse and inspect the data
        data = io.StringIO(response.text)
        df = pd.read_csv(data)

        # Check if active fire data was actually retrieved
        if df.empty:
            print("⚠️ WARNING: Request successful, but no active fires were detected in this area/timeframe.")
            return None

        print(f"🎉 Successfully retrieved {len(df)} active fire records from API.")
        return df

    except requests.exceptions.HTTPError as errh:
        print(f"❌ HTTP Error (Check API key or rate limits): {errh}")
    except requests.exceptions.ConnectionError as errc:
        print(f"❌ Connection Error: {errc}")
    except requests.exceptions.Timeout as errt:
        print(f"❌ Timeout Error: {errt}")
    except pd.errors.EmptyDataError:
        print("❌ Data Parsing Error: NASA returned invalid CSV format.")
    except Exception as err:
        print(f"❌ Unexpected error occurred during fetch: {err}")

    return None

## Save the provided DataFrame to a local CSV file. Only handles file I/O operations.

In [ ]:
def save_to_csv_files(df, base_filename):

    if df is None or df.empty:
        print("⚠️ No data to save.")
        return

    csv_name = f"{base_filename}.csv"

    try:
        # Save as CSV
        df.to_csv(csv_name, index=False, encoding='utf-8')
        print(f"✅ Saved CSV: {csv_name}")

    except Exception as e:
        print(f"❌ Error occurred while saving files: {e}")

## Save the provided DataFrame to a local Shapefile. Only handles file I/O operations.

In [ ]:
def save_to_shp_files(df, base_filename):
    if df is None or df.empty:
        print("⚠️ No data to save.")
        return

    shp_name = f"{base_filename}.shp"

    try:
        # Save as Shapefile
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        )
        gdf.to_file(shp_name, driver="ESRI Shapefile")
        print(f"✅ Saved Shapefile: {shp_name}")

    except Exception as e:
        print(f"❌ Error occurred while saving files: {e}")

## Retrieve data from FIRMS

In [ ]:
# 2.1 Use the historical fetch function for date ranges.
# Skip this optional section when no FIRMS key is configured.
if MAP_KEY:
    fire_df = fetch_historical_firms_data(
        map_key=MAP_KEY,
        source=SOURCE,
        bbox=BBOX,
        start_date_str=START_DATE,
        end_date_str=END_DATE
    )
else:
    print("ℹ️ FIRMS download skipped. Add FIRMS_MAP_KEY in Colab Secrets to enable it.")
    fire_df = None


## Generate shapefile & CSV file

In [ ]:
# 3. If data was successfully fetched, save it to CSV and Shapefile
if fire_df is not None:
    file_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # base_name = f"firms_data_{file_timestamp}"
    base_name = f"firms_data_historical_{START_DATE}_to_{END_DATE}_{file_timestamp}"
    save_to_csv_files(fire_df, base_name)
    save_to_shp_files(fire_df, base_name)

# Calculate dNBR and display on map

In [ ]:
!pip install geemap -q                                                          # install geemap library
!pip install ffmpeg-python -q                                                   # required by geemap timelapse

In [ ]:
import ee                                                                       # Google Earth Engine API
import geemap                                                                   # interactive GEE maps
import folium                                                                   # interactive maps
import time                                                                     # time
import numpy as np                                                              # numerical arrays
import pandas as pd                                                             # data tables
import geopandas as gpd                                                         # geospatial dataframes
import matplotlib.pyplot as plt                                                 # plotting
import matplotlib.patches as mpatches                                           # legend patches
import matplotlib.colors as mcolors                                             # colour utilities
from matplotlib.gridspec import GridSpec                                        # subplot layout
from matplotlib.colors import LinearSegmentedColormap                           # custom colourmaps
from IPython.display import display, HTML                                       # notebook display
import ipywidgets as widgets                                                    # import widgets
from IPython.display import clear_output, display
import time
from datetime import datetime, timedelta

print("All libraries loaded!")                                                  # Confirm success

In [ ]:
try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)


In [ ]:
print(f"✅ GEE initialized successfully with project: {GEE_PROJECT_ID}")


In [ ]:
Map = geemap.Map()
roi = ee.Geometry.BBox(*[float(x) for x in BBOX.split(',')])
Map.centerObject(roi, 8)

In [ ]:
# 1. Set time windows based on the fire dates.
# Earth Engine filterDate uses [start, end): the end date is exclusive.
offset_days = 30
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
end_dt = datetime.strptime(END_DATE, "%Y-%m-%d")
if start_dt > end_dt:
    raise ValueError("START_DATE cannot be later than END_DATE")

prefire_start = (start_dt - timedelta(days=offset_days)).strftime("%Y-%m-%d")
prefire_end = START_DATE
postfire_start = (end_dt + timedelta(days=1)).strftime("%Y-%m-%d")
postfire_end = (end_dt + timedelta(days=offset_days + 1)).strftime("%Y-%m-%d")

# 2. Load Sentinel-2 Surface Reflectance (SR) data.
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

def add_nbr(image):
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

def mask_s2_clouds(image):
    """Mask invalid, cloud-shadow, cloud, cirrus, and snow/ice SCL pixels."""
    scl = image.select('SCL')
    invalid = (scl.eq(0)
               .Or(scl.eq(1))
               .Or(scl.eq(3))
               .Or(scl.eq(7))
               .Or(scl.eq(8))
               .Or(scl.eq(9))
               .Or(scl.eq(10))
               .Or(scl.eq(11)))
    return image.updateMask(invalid.Not())

def prepare_s2_collection(start_date_str, end_date_str):
    """Create a date-filtered collection with scene- and pixel-level cloud masks."""
    return (s2.filterBounds(roi)
            .filterDate(start_date_str, end_date_str)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .map(mask_s2_clouds))

def get_median_composite(start_date_str, end_date_str, with_nbr=False,
                         allow_empty=False):
    """Return the median composite and source-image count for one period."""
    collection = prepare_s2_collection(start_date_str, end_date_str)
    image_count = int(collection.size().getInfo())
    if image_count == 0:
        if allow_empty:
            return None, 0
        raise RuntimeError(
            f"No Sentinel-2 images found from {start_date_str} to "
            f"{end_date_str} with CLOUDY_PIXEL_PERCENTAGE < 20."
        )
    if with_nbr:
        collection = collection.map(add_nbr)
    return collection.median().clip(roi), image_count

def classify_dnbr(dnbr_image):
    """Classify continuous dNBR using USGS/FIREMON thresholds."""
    return (ee.Image(0)
            .where(dnbr_image.lt(-0.25), 1)
            .where(dnbr_image.gte(-0.25).And(dnbr_image.lt(-0.1)), 2)
            .where(dnbr_image.gte(-0.1).And(dnbr_image.lt(0.1)), 3)
            .where(dnbr_image.gte(0.1).And(dnbr_image.lt(0.27)), 4)
            .where(dnbr_image.gte(0.27).And(dnbr_image.lt(0.44)), 5)
            .where(dnbr_image.gte(0.44).And(dnbr_image.lt(0.66)), 6)
            .where(dnbr_image.gte(0.66), 7)
            .updateMask(dnbr_image.mask()))

print(
    f"⏳ Computing NBR composites for {prefire_start} to {prefire_end} "
    f"and {postfire_start} to {postfire_end}"
)


In [ ]:
# 3. Build pre-fire and first post-fire 30-day NBR composites.
pre_fire_img, pre_image_count = get_median_composite(
    prefire_start, prefire_end, with_nbr=True
)
post_fire_img, post_image_count = get_median_composite(
    postfire_start, postfire_end, with_nbr=True
)
print(f"✅ Source images: pre-fire={pre_image_count}, post-fire={post_image_count}")

# 4. dNBR = pre-fire NBR - post-fire NBR.
dnbr = pre_fire_img.select('NBR').subtract(
    post_fire_img.select('NBR')
).rename('dNBR')

# 5. Classify dNBR for visualization.
classified_dnbr = classify_dnbr(dnbr)

vis_dnbr = {
    'min': 1,
    'max': 7,
    'palette': [
        '006400', # 1: Dark Green (Significant recovery)
        '90EE90', # 2: Light Green (Slight recovery)
        'E0E0E0', # 3: Light Grey (Unburned)
        'FFFF00', # 4: Yellow (Low severity burn)
        'FFA500', # 5: Orange (Moderate-low severity burn)
        'FF0000', # 6: Red (Moderate-high severity burn)
        '800080'  # 7: Purple (High severity burn)
    ]
}

Map.addLayer(classified_dnbr, vis_dnbr, f'dNBR (Classified) - {START_DATE}')
Map


## Remove water & Ocean Masking

In [ ]:
# ==========================================
# Remove ocean and inland-water interference
# ==========================================
print("🌊 Applying land boundary and water masks...")

land_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
jrc_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
inland_water_mask = jrc_water.select('max_extent').eq(0)

# Apply masks to the continuous dNBR image before classification.
dnbr_land = dnbr.clipToCollection(land_boundaries)
dnbr_clean = dnbr_land.updateMask(inland_water_mask)
dnbr_no_water = classify_dnbr(dnbr_clean)

Map.addLayer(dnbr_no_water, vis_dnbr, f'dNBR (Cleaned) - {START_DATE}')
print("✅ Ocean and water noise removed!")


## Add Legend on map if it does not have one

In [ ]:
def add_unique_legend(map_obj, legend_dict, title, position, unique_tag):
    """
    Adds a legend to a geemap map, preventing duplicate additions.

    Parameters:
    map_obj: geemap.Map instance
    legend_dict: Dictionary containing text and colors
    title: Legend title
    position: Legend position on the map ('bottomright', 'bottomleft', etc.)
    unique_tag: Unique string tag to prevent duplication (e.g., 'dnbr_legend_added')
    """
    # Dynamically check if the map object already has this specific tag
    if not hasattr(map_obj, unique_tag):
        map_obj.add_legend(
            title=title,
            legend_dict=legend_dict,
            position=position
        )
        # Dynamically add the tag to the object with a value of True
        setattr(map_obj, unique_tag, True)
        print(f"✅ Legend 【{title}】 successfully loaded!")
    else:
        print(f"ℹ️ Legend 【{title}】 already exists, skipping duplicate addition.")

In [ ]:
legend_dict_dnbr = {
    "Significant vegetation recovery (< -0.25)": "006400",
    "Slight vegetation recovery (-0.25 to -0.1)": "90EE90",
    "Unburned/No significant change (-0.1 to 0.1)": "E0E0E0",
    "Low severity burn (0.1 to 0.27)": "FFFF00",
    "Moderate-low severity burn (0.27 to 0.44)": "FFA500",
    "Moderate-high severity burn (0.44 to 0.66)": "FF0000",
    "High severity burn (> 0.66)": "800080"
}

add_unique_legend(
    map_obj=Map,
    legend_dict=legend_dict_dnbr,
    title="dNBR Burn Severity",
    position="bottomright",
    unique_tag="dnbr_legend"
)

print("✅ dNBR Legend loaded (duplicate protection enabled)!")
Map

## Fire Mask Thresholding

In [ ]:
Map1 = geemap.Map()
Map1.centerObject(roi, 8)

print("🔥 Extracting burned areas with continuous dNBR > 0.1...")

# Create the fixed burn-scar mask from continuous dNBR values. Do not compare
# the 1-7 class codes, because that includes or excludes the wrong classes.
burned_mask = dnbr_clean.gt(0.1)
burned_area_only = dnbr_no_water.updateMask(burned_mask)

vis_burned_only = {
    'min': 4,
    'max': 7,
    'palette': ['FFFF00', 'FFA500', 'FF0000', '800080']
}

Map1.addLayer(
    burned_area_only,
    vis_burned_only,
    'Burned Area Only (dNBR > 0.1)'
)

legend_dict_dnbr = {
    "Low severity burn (0.1 - 0.27)": "FFFF00",
    "Moderate-low severity burn (0.27 - 0.44)": "FFA500",
    "Moderate-high severity burn (0.44 - 0.66)": "FF0000",
    "High severity burn (> 0.66)": "800080"
}

add_unique_legend(
    map_obj=Map1,
    legend_dict=legend_dict_dnbr,
    title="dNBR Burn Severity",
    position="bottomright",
    unique_tag="dnbr_legend"
)
print("✅ Burned-area layer generated; unburned areas and water are transparent.")
Map1


## Monthly dNBR change

In [ ]:
Map2 = geemap.Map()
Map2.centerObject(roi, 8)

months_to_track = 12
window_days = 30
print("🌱 Starting monthly post-fire dNBR monitoring...")

for i in range(months_to_track):
    # Month 1 is day 1-30 after the fire; filterDate end is exclusive.
    period_start_dt = end_dt + timedelta(days=1 + i * window_days)
    period_end_exclusive_dt = period_start_dt + timedelta(days=window_days)
    period_end_inclusive_dt = period_end_exclusive_dt - timedelta(days=1)

    p_start_str = period_start_dt.strftime("%Y-%m-%d")
    p_end_str = period_end_exclusive_dt.strftime("%Y-%m-%d")
    display_end_str = period_end_inclusive_dt.strftime("%Y-%m-%d")
    print(f"Month {i+1}: {p_start_str} to {display_end_str}")

    recovery_img, recovery_count = get_median_composite(
        p_start_str, p_end_str, with_nbr=True, allow_empty=True
    )
    if recovery_img is None:
        print(f"⚠️ Month {i+1} skipped: no qualifying Sentinel-2 images.")
        continue

    # Compare every month against the same pre-fire baseline and retain the
    # same initial burned-area footprint, including pixels that later recover.
    recovery_dnbr = pre_fire_img.select('NBR').subtract(
        recovery_img.select('NBR')
    ).rename('dNBR')
    recovery_dnbr_clean = (recovery_dnbr
                           .clipToCollection(land_boundaries)
                           .updateMask(inland_water_mask)
                           .updateMask(burned_mask))
    classified_recovery = classify_dnbr(recovery_dnbr_clean)

    layer_name = (
        f"Post-Fire Month {i+1} dNBR "
        f"({p_start_str} to {display_end_str})"
    )
    Map2.addLayer(classified_recovery, vis_dnbr, layer_name, i == 0)

print("✅ Available monthly dNBR layers loaded. Use Layer Control to compare them.")


In [ ]:
# ==========================================
# Add dNBR (Burn Severity) standard academic legend (duplicate protected)
# ==========================================

legend_dict_dnbr_full = {
    "Significant vegetation recovery (< -0.25)": "006400",
    "Slight vegetation recovery (-0.25 to -0.1)": "90EE90",
    "Unburned/No significant change (-0.1 to 0.1)": "E0E0E0",
    "Low severity burn (0.1 to 0.27)": "FFFF00",
    "Moderate-low severity burn (0.27 to 0.44)": "FFA500",
    "Moderate-high severity burn (0.44 to 0.66)": "FF0000",
    "High severity burn (> 0.66)": "800080"
}

add_unique_legend(
    map_obj=Map2,
    legend_dict=legend_dict_dnbr_full,
    title="dNBR Burn Severity (USGS Full Version)",
    position="bottomright",
    unique_tag="dnbr_full_legend" # Updated duplicate protection tag name to ensure successful addition
)
print("✅ dNBR Legend loaded (duplicate protection enabled)!")

print("✅ dNBR Layer loaded!")
Map2

## Monthly NDVI change

In [ ]:
Map3 = geemap.Map()
Map3.centerObject(roi, 8)

print("🗺️ Rendering burned-area NDVI evolution for selected post-fire months...")

def get_ndvi(image):
    return image.normalizedDifference(['B8', 'B4']).rename('NDVI')

vis_ndvi = {
    'min': 0.0,
    'max': 0.8,
    'palette': ['A52A2A', 'FFA500', 'FFFF00', '00FF00', '008000']
}

target_months = [1, 3, 6, 9, 12]

for m in target_months:
    start_date_dt = end_dt + timedelta(days=1 + (m - 1) * 30)
    end_date_exclusive_dt = start_date_dt + timedelta(days=30)
    end_date_inclusive_dt = end_date_exclusive_dt - timedelta(days=1)

    s_str = start_date_dt.strftime("%Y-%m-%d")
    e_str = end_date_exclusive_dt.strftime("%Y-%m-%d")
    display_end_str = end_date_inclusive_dt.strftime("%Y-%m-%d")

    monthly_img, monthly_count = get_median_composite(
        s_str, e_str, allow_empty=True
    )
    if monthly_img is None:
        print(f"⚠️ NDVI map month {m} skipped: no qualifying Sentinel-2 images.")
        continue

    clean_ndvi_burned_only = (get_ndvi(monthly_img)
                              .clipToCollection(land_boundaries)
                              .updateMask(inland_water_mask)
                              .updateMask(burned_mask))

    layer_name = (
        f"Post-Fire Month {m} Burned Area NDVI "
        f"({s_str} to {display_end_str})"
    )
    Map3.addLayer(clean_ndvi_burned_only, vis_ndvi, layer_name, m == 1)

legend_dict_ndvi = {
    "Water/Severe burn scar (< 0.0)": "A52A2A",
    "Bare land/Sparse herbaceous (0.0 - 0.2)": "FFA500",
    "Low density vegetation/Shrubs (0.2 - 0.4)": "FFFF00",
    "Medium density forest (0.4 - 0.6)": "00FF00",
    "High density healthy forest (> 0.6)": "008000"
}

add_unique_legend(
    map_obj=Map3,
    legend_dict=legend_dict_ndvi,
    title="NDVI Vegetation Recovery Index",
    position="bottomright",
    unique_tag="ndvi_legend"
)
print("✅ Burned-area NDVI time-series layers loaded.")
Map3


## NDVI change curve

In [ ]:
import matplotlib.pyplot as plt

# 1. Calculate the pre-fire NDVI baseline within the fixed burned area.
pre_ndvi = get_ndvi(pre_fire_img)
pre_ndvi_burned_only = pre_ndvi.updateMask(burned_mask)

print("⏳ Calculating pre-fire NDVI baseline value...")
pre_ndvi_val = pre_ndvi_burned_only.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=100,
    maxPixels=1e10
).get('NDVI').getInfo()

if pre_ndvi_val is None:
    print("⚠️ No valid pre-fire NDVI baseline pixels were found.")
else:
    print(f"🔥 Average pre-fire NDVI in the burned area: {pre_ndvi_val:.4f}")

# 2. Extract the mean burned-area NDVI for 12 post-fire 30-day periods.
months_to_track = 12
dates = []
ndvi_values = []
print("📊 Extracting the 12-month post-fire NDVI time series...")

for i in range(months_to_track):
    period_start_dt = end_dt + timedelta(days=1 + i * 30)
    period_end_exclusive_dt = period_start_dt + timedelta(days=30)
    period_end_inclusive_dt = period_end_exclusive_dt - timedelta(days=1)

    p_start_str = period_start_dt.strftime("%Y-%m-%d")
    p_end_str = period_end_exclusive_dt.strftime("%Y-%m-%d")
    display_end_str = period_end_inclusive_dt.strftime("%Y-%m-%d")

    monthly_img, monthly_count = get_median_composite(
        p_start_str, p_end_str, allow_empty=True
    )
    if monthly_img is None:
        print(f"⚠️ Month {i+1} skipped: no qualifying Sentinel-2 images.")
        continue

    monthly_ndvi = get_ndvi(monthly_img).updateMask(burned_mask)
    try:
        val = monthly_ndvi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=100,
            maxPixels=1e10
        ).get('NDVI').getInfo()

        if val is not None:
            dates.append(p_start_str)
            ndvi_values.append(val)
            print(
                f"✅ Month {i+1} ({p_start_str} to {display_end_str}): "
                f"NDVI = {val:.4f}"
            )
        else:
            print(f"⚠️ Month {i+1}: no valid NDVI pixels after masking.")
    except Exception as exc:
        print(f"❌ Month {i+1} ({p_start_str}) extraction failed: {exc}")

# 3. Plot the recovery trajectory.
if ndvi_values:
    df_ndvi = pd.DataFrame({'Date': pd.to_datetime(dates), 'NDVI': ndvi_values})
    plt.figure(figsize=(12, 6), dpi=100)
    plt.plot(
        df_ndvi['Date'], df_ndvi['NDVI'], marker='o', linestyle='-',
        color='forestgreen', linewidth=2.5, markersize=8,
        label='Post-fire NDVI Trajectory'
    )

    if pre_ndvi_val is not None:
        plt.axhline(
            y=pre_ndvi_val, color='red', linestyle='--', linewidth=2,
            label=f'Pre-fire Baseline ({pre_ndvi_val:.4f})'
        )

    plt.title('Post-Fire Vegetation Recovery Trajectory (12 Months)',
              fontsize=16, fontweight='bold')
    plt.xlabel('Evaluation Date', fontsize=12)
    plt.ylabel('Mean NDVI within Burned Area', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='lower right', fontsize=11)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data collected to plot the chart.")
